# Lab 18: Model Evaluation — Diagnosing a Metric That Chose Itself
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 120 min Core + 20 min Extension

---

**Learning Objectives:**
- Score three models four ways on one test set, and say what each metric measures that makes it choose the model it does
- Put a bootstrap interval on every evaluation number, and see how its width depends on the number of test rows
- Use decision curve analysis, in which a threshold stands for a cost ratio, so that nobody has to name a dollar figure
- Write the interval that settles a ranking: a paired bootstrap of the gap between two models scored on the same rows
- Ship a module in which every comparison of two models comes with a paired interval

**Dataset:** 10,000 simulated rows with a known outcome model (17.8% of outcomes are 1s)

**Format:** Each diagnosis shows you a result that is wrong, or wrong-looking, and asks what happened. Write your answers in the text cell under each set of questions.

**Deliverable:** your written diagnoses (Diagnoses 1, 2 and 3, and the question in Ship It), the blank you fill in Diagnoses 2 and 3, the required `paired_gap()` function, and `src/evaluation.py`, a module you finish with your `paired_gap()`, save and upload.

**Time:** Core (120 min, including the required Write it yourself) + Extension (optional, 20 min)

With setup, the README and the GitHub submission, plan on about two and a quarter hours, most of it after class. The optional Extension adds about 20 minutes.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Diagnosis 2:** one blank, the call that puts an interval on the forest's ROC AUC for a smaller test set.
> - **Diagnosis 3:** one blank, the threshold that a 10 : 1 cost ratio implies.
> - **Write it yourself (required, after Diagnosis 3):** `paired_gap()`, a bootstrap interval for the gap between two models scored on the same rows, about ten lines in an empty cell. A check cell tells you when it is right. In Ship It you paste it, unchanged, into the module.
> - **Extension (optional):** a test set from a shifted population, from a three-step outline.
>
> **You write, in text cells:** three answers in Diagnosis 1, three in Diagnosis 2, three in Diagnosis 3 and one in Ship It.
>
> **You run and read:** everything else. That is the simulated data and the three models, the four metrics, the bootstrap function and the table of intervals, the net benefit function and the decision curve, and the rest of Ship It's module, `evaluation.py`, which is working code to read, not to retype. Read each cell before you run it: the questions ask about what it prints, and Write it yourself changes Diagnosis 2's bootstrap function so that it resamples two models together.

## Setup

The first line of the setup cell installs scikit-learn 1.8, 1.9 or 1.10: the expected numbers in
this lab were produced on 1.8 and 1.9, and 1.11 removes an argument the boosted model uses. The cell
prints your scikit-learn version. If it stops with an `ImportError`, or shows another version, use
**Runtime → Restart session** and run this cell again. The `warnings.filterwarnings` line hides one notice from scikit-learn 1.9; Diagnosis
1 explains it where the boosted model is built.

In [ ]:
# GIVEN — run as-is
# Setup: the libraries and the seed
!pip install "scikit-learn>=1.8,<1.11" -q

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn import metrics
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Path("src").mkdir(exist_ok=True)     # Ship It writes its module into this folder
# scikit-learn 1.9 says criterion= is no longer needed; the boosted model sets it so 1.8 gives the same model
warnings.filterwarnings("ignore", message="The parameter `criterion` is deprecated")

RANDOM_STATE = 42                    # the seed for the data, the split, the models and every bootstrap
print(f"scikit-learn {sklearn.__version__} (this lab needs 1.8, 1.9 or 1.10)")
print("Setup complete.")

## Diagnosis 1: Four Metrics, Three Winners (23 min)

A team reports that their new model "beats the baseline". They chose the metric after seeing the
results, without noticing that they had. This diagnosis repeats their evaluation: three models, four
metrics, one test set.

The data are simulated, so you know how they were made: 10,000 rows, eight features, and an outcome
whose log-odds are linear in the features except in one region. Where the first feature is above
0.5 and the second is below 0, the log-odds jump by 2. A logistic regression cannot represent that
jump, and trees can. Without it the logistic model would be the true model and would win every
metric.

**Reading the code.** This is Lab 17's simulation with eight columns. `(X[:, 0] > 0.5) & (X[:, 1] < 0)`
is True for the rows in the region (`&` is "and", row by row), and True counts as 1 when it is
multiplied by 2.0. The mean of the 0s and 1s is the *prevalence*, the share of 1s.

In [ ]:
# GIVEN — run as-is
# 10,000 simulated rows: eight features, and outcomes drawn from log-odds with one non-linear region
rng = np.random.default_rng(RANDOM_STATE)
n = 10_000
X = rng.normal(0, 1, (n, 8))
region = (X[:, 0] > 0.5) & (X[:, 1] < 0)          # the region a linear model cannot represent
lin = -2.8 + X @ np.array([1.1, -0.9, 0.7, 0.4, 0.0, 0.0, -0.3, 0.2]) + 2.0 * region
y = rng.binomial(1, 1 / (1 + np.exp(-lin)))
print(f"prevalence = {y.mean():.1%}")

**Expected output:** `prevalence = 17.8%`.

**Reading the code.** `train_test_split(..., test_size=0.3, stratify=y)` holds out 30% of the rows,
3,000, as a test set, with the same share of 1s in both parts. The three models are fitted on the
training rows only:
- `logistic`: the features standardised, then a logistic regression (`max_iter` lets the solver take more steps).
- `forest`: 250 trees; `min_samples_leaf=10` makes every leaf hold at least 10 training rows.
- `boosted`: 300 small trees (`max_depth=3`), each fitted to the errors the trees before it left.
  `criterion="squared_error"` is how scikit-learn 1.9 always scores a split; 1.8's default differs
  slightly, so naming it gives the same model on both (setup hid 1.9's warning that it is not needed).

`for name, model in models.items()` goes through the dictionary's name-and-model pairs, and `probs`
keeps each model's test probabilities by name. Fitting takes about ten seconds.

In [ ]:
# GIVEN — run as-is
# Hold out 30% of the rows; fit three models on the rest and keep each one's test probabilities
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=RANDOM_STATE, stratify=y)
models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "forest": RandomForestClassifier(n_estimators=250, min_samples_leaf=10,
                                     random_state=RANDOM_STATE),
    "boosted": GradientBoostingClassifier(n_estimators=300, max_depth=3, criterion="squared_error",
                                          random_state=RANDOM_STATE),
}
probs = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probs[name] = model.predict_proba(X_test)[:, 1]
print(f"test rows: {len(y_test)}, of which {y_test.sum()} are 1s")

**Expected output:** `test rows: 3000, of which 533 are 1s`.

**Four ways to score the same probabilities.**
- **Accuracy at 0.5:** call a row a 1 when its probability is at least 0.5, and count the share of rows called right.
- **ROC AUC:** the chance that a randomly chosen 1 gets a higher probability than a randomly chosen 0 (Lab 17).
- **Average precision (AP):** the area under the precision-recall curve. Like AUC it depends only on the order of the probabilities, but its no-skill value is the prevalence rather than 0.5.
- **Minimum expected cost:** here a missed 1 (a false negative) costs 10 and a false alarm (a false positive) costs 1. `cost_per_row` charges those costs for the rule "flag every row whose probability is at least t" and divides by the number of rows. `min_cost` tries 200 thresholds between 0.01 and 0.9 and keeps the cheapest. Lower is better.

**Reading the code.** `cost_per_row` counts its errors as Lab 17's `cost_per_1000` did (`~` is
"not"). `np.linspace(0.01, 0.9, 200)` is 200 evenly spaced thresholds, and
`min(cost_per_row(...) for t in COST_T)` keeps the smallest of their costs.

In [ ]:
# GIVEN — run as-is
# Accuracy at 0.5, and the expected cost of a threshold rule when a missed 1 costs 10 times a false alarm
FN_COST, FP_COST = 10.0, 1.0
COST_T = np.linspace(0.01, 0.9, 200)          # the thresholds min_cost tries


def accuracy_at_half(y_true, y_prob):
    """Share of rows called right when every probability of at least 0.5 is called a 1."""
    return metrics.accuracy_score(y_true, y_prob >= 0.5)


def cost_per_row(y_true, y_prob, t, fn_cost=FN_COST, fp_cost=FP_COST):
    """Average cost per row of flagging every row with a probability of at least t."""
    flag = y_prob >= t
    missed = np.sum(~flag & (y_true == 1))          # false negatives: 1s not flagged
    false_alarms = np.sum(flag & (y_true == 0))     # false positives: 0s flagged
    return (fn_cost * missed + fp_cost * false_alarms) / len(y_true)


def min_cost(y_true, y_prob):
    """The lowest cost_per_row over the 200 thresholds in COST_T."""
    return min(cost_per_row(y_true, y_prob, t) for t in COST_T)

**Reading the code.** `METRICS` is a dictionary whose values are functions, stored by name without
parentheses, so not yet called; `fn(y_test, p)` calls one. `"".join(...)` glues the four formatted
numbers into one line. The later diagnoses use `METRICS` again.

In [ ]:
# GIVEN — run as-is
# The four metrics for each model, on the same 3,000 test rows
METRICS = {"accuracy": accuracy_at_half, "ROC AUC": metrics.roc_auc_score,
           "AP": metrics.average_precision_score, "min cost": min_cost}
print(f"{'model':<10}" + "".join(f"{metric:>10}" for metric in METRICS))
for name, p in probs.items():
    print(f"{name:<10}" + "".join(f"{fn(y_test, p):>10.4f}" for fn in METRICS.values()))

**Expected output:**
```
model       accuracy   ROC AUC        AP  min cost
logistic      0.8763    0.8898    0.6966    0.4347
forest        0.8897    0.8981    0.7152    0.4370
boosted       0.8923    0.8902    0.7189    0.4667
```
Higher is better in the first three columns, lower in the last.

### YOUR DIAGNOSIS

1. How many different models could the team have reported as the winner, and why? Say what each of the four columns measures that makes it pick the model it picks.
2. Accuracy at 0.5 is the weakest of the four. Name its two separate problems on this data, using the prevalence.
3. The team chose the metric after seeing the results. Name this failure and its analogue in Lab 15, propose in one sentence the rule that prevents it, and say what that rule costs.

*Your answers here:*

---

## Diagnosis 2: A Ranking Without an Interval (27 min)

Every number in that table comes from one test set of 3,000 rows. Another 3,000 rows from the same
population would give other numbers, and might rank the models differently. The bootstrap from Lab 7
measures how much: draw 3,000 rows from the test set *with replacement*, so that some rows appear
twice and some not at all; score the model on that resample; repeat 1,000 times; and take the middle
95% of the 1,000 scores, from the 2.5th to the 97.5th percentile, as the interval.

**Reading the code.** The metric is an argument, `metric_fn`, passed like the functions in
`METRICS`. `np.random.default_rng(seed)` makes a generator of its own, so a seed always gives the same
resamples. `gen.integers(0, n, n)` draws n row positions from 0 to n − 1 with replacement, and
`y_true[rows]` and `y_prob[rows]` keep those rows, a row drawn twice appearing twice. `for _ in
range(boots)` repeats the body `boots` times (`_` says the loop number is not used).
`np.percentile(values, [2.5, 97.5])` gives the two cut points. The function returns three numbers,
which `value, low, high = metric_interval(...)` unpacks into three names.

In [ ]:
# GIVEN — run as-is
# A bootstrap interval for any metric: resample the test rows, score again, keep the middle 95%
def metric_interval(y_true, y_prob, metric_fn, boots=1000, seed=RANDOM_STATE):
    """The metric on all rows, and a 95% percentile bootstrap interval for it."""
    gen = np.random.default_rng(seed)
    n = len(y_true)
    values = []
    for _ in range(boots):
        rows = gen.integers(0, n, n)          # n row positions, drawn with replacement
        values.append(metric_fn(y_true[rows], y_prob[rows]))
    low, high = np.percentile(values, [2.5, 97.5])
    return metric_fn(y_true, y_prob), low, high

The next cell puts an interval on each of the twelve numbers in Diagnosis 1's table, with the same
`METRICS`. That is 12,000 resamples, so it takes about half a minute; most of the time goes to
`min cost`, which tries 200 thresholds on every resample.

In [ ]:
# GIVEN — run as-is
# Diagnosis 1's table again, each number with its 95% bootstrap interval
for metric, fn in METRICS.items():
    for name, p in probs.items():
        value, low, high = metric_interval(y_test, p, fn)
        print(f"{metric:<9}{name:<10}{value:.4f}  [{low:.4f}, {high:.4f}]")

**Expected output:**
```
accuracy logistic  0.8763  [0.8650, 0.8877]
accuracy forest    0.8897  [0.8783, 0.9007]
accuracy boosted   0.8923  [0.8800, 0.9030]
ROC AUC  logistic  0.8898  [0.8750, 0.9042]
ROC AUC  forest    0.8981  [0.8837, 0.9121]
ROC AUC  boosted   0.8902  [0.8760, 0.9047]
AP       logistic  0.6966  [0.6594, 0.7322]
AP       forest    0.7152  [0.6765, 0.7554]
AP       boosted   0.7189  [0.6836, 0.7546]
min cost logistic  0.4347  [0.3870, 0.4710]
min cost forest    0.4370  [0.3813, 0.4710]
min cost boosted   0.4667  [0.4173, 0.4997]
```

**Your task.** How wide an interval is depends on how many test rows there are. Fill in the blank
with the call that gives the forest's ROC AUC and its interval on only the first m test rows, using
`metric_interval`. `a[:m]` keeps the first m entries of an array.

In [ ]:
# YOUR TASK — Diagnosis 2: fill in the blank (____)
for m in [750, 1500, 3000]:
    auc_m, low_m, high_m = ____          # the forest's ROC AUC and its interval, on the first m test rows
    print(f"first {m:>4} rows: ROC AUC {auc_m:.4f}  [{low_m:.4f}, {high_m:.4f}]  "
          f"width {high_m - low_m:.4f}")

**Expected output:**
```
first  750 rows: ROC AUC 0.9171  [0.8908, 0.9393]  width 0.0485
first 1500 rows: ROC AUC 0.8999  [0.8781, 0.9189]  width 0.0408
first 3000 rows: ROC AUC 0.8981  [0.8837, 0.9121]  width 0.0284
```
The last line must repeat the forest's ROC AUC line from the table above; if it does not, check the
model and the metric. Three identical lines mean the rows were not cut to m. An `IndexError`, or a
`ValueError` about inconsistent numbers of samples, means the outcomes and the probabilities were cut
differently: cut both.

### YOUR DIAGNOSIS

1. Diagnosis 1's margins are small: 0.8923 against 0.8897 on accuracy, 0.8981 against 0.8902 on ROC AUC. Compare them with the widths of the intervals in the table. From these intervals alone, could you defend either of those two winners?
2. From your three widths: by about how much does the width shrink when the test set doubles from 1,500 to 3,000 rows, and what rule does that suggest? Use the rule to estimate how many test rows it would take for the forest's ROC AUC interval to be as narrow as its 0.0079 lead over the boosted model.
3. The forest's and the boosted model's ROC AUC intervals overlap almost completely. Does that show their AUCs could be equal? Think about a resample that happens to draw many hard rows: what does it do to both models' scores at once, and what does that mean for an interval on the gap between them?

*Your answers here:*

---

## Diagnosis 3: Costs You Cannot Get (25 min)

The min cost column needed a cost ratio, 10 : 1. Ask a business for one and you will usually get a
shrug. Decision curve analysis (Vickers and Elkin, 2006, from class) works around that. Someone who
acts on every row whose probability is at least t has said, by choosing t, that a false alarm costs
t / (1 − t) times a missed 1: at a probability of exactly t, acting and not acting have the same
expected cost. So a threshold stands for a cost ratio. The **net benefit** of acting at t counts the
true positives and charges each false positive at that rate:

NB(t) = TP/n − (FP/n) × t / (1 − t)

It has two references: *treat none*, which flags nobody and has a net benefit of 0, and *treat all*,
which flags everyone. A model is worth consulting at threshold t only if its net benefit beats both.

**Reading the code.** `net_benefit` counts the flagged 1s (true positives) and the flagged 0s
(false positives) the way `cost_per_row` counted its errors.

In [ ]:
# GIVEN — run as-is
# Net benefit of flagging every row with a probability of at least t
def net_benefit(y_true, y_prob, t):
    """True positives per row, minus false positives per row at the exchange rate t / (1 - t)."""
    flag = y_prob >= t
    n = len(y_true)
    true_pos = np.sum(flag & (y_true == 1))
    false_pos = np.sum(flag & (y_true == 0))
    return true_pos / n - false_pos / n * t / (1 - t)

**Your task.** Diagnosis 1 charged 10 for a missed 1 and 1 for a false alarm. Fill in the blank with
the threshold at which those costs make acting and not acting equally expensive, computed from
`FN_COST` and `FP_COST` rather than typed as a number. The cell then prints, at that threshold, each
model's net benefit and its cost per row from Diagnosis 1, and the net benefit of treating everyone.
`np.ones(len(y_test))` is an array of 1s: a "model" that gives every row a probability of 1, and so
flags every row.

In [ ]:
# YOUR TASK — Diagnosis 3: fill in the blank (____)
T_RULE = ____                    # the threshold a 10 : 1 cost ratio implies, from FN_COST and FP_COST
print(f"T_RULE = {T_RULE:.4f}")
for name, p in probs.items():
    print(f"{name:>10}: net benefit {net_benefit(y_test, p, T_RULE):.4f}   "
          f"cost per row {cost_per_row(y_test, p, T_RULE):.4f}")
print(f"{'treat all':>10}: net benefit {net_benefit(y_test, np.ones(len(y_test)), T_RULE):.4f}")

**Expected output:**
```
T_RULE = 0.0909
  logistic: net benefit 0.1339   cost per row 0.4373
    forest: net benefit 0.1327   cost per row 0.4497
   boosted: net benefit 0.1277   cost per row 0.5000
 treat all: net benefit 0.0954
```
`T_RULE = 0.1000` and `T_RULE = 0.9091` are the two likely slips; Python runs both without an
error.

**The decision curve.** The next cell computes each model's net benefit at the thresholds 0.01,
0.02, ..., 0.50 and plots it with the two references. `{name: [...] for name, p in probs.items()}`
builds a dictionary with one list of net benefits per model, and `ax.axhline(0, ...)` draws treat
none. The y-axis stops at −0.02, so treat all leaves the plot at the higher thresholds.

In [ ]:
# GIVEN — run as-is
# Decision curve: net benefit at thresholds 0.01 to 0.50 for the three models and the two references
DCA_T = np.round(np.arange(0.01, 0.51, 0.01), 2)
treat_all = np.ones(len(y_test))
curves = {name: [net_benefit(y_test, p, t) for t in DCA_T] for name, p in probs.items()}
curves["treat all"] = [net_benefit(y_test, treat_all, t) for t in DCA_T]

fig, ax = plt.subplots(figsize=(7, 4.5))
for name, nb in curves.items():
    ax.plot(DCA_T, nb, label=name)
ax.axhline(0, color="gray", linestyle="--", label="treat none")
ax.set_ylim(-0.02, 0.2)
ax.set_xlabel("threshold probability t")
ax.set_ylabel("net benefit")
ax.legend()
plt.show()

**Expected output:** three model curves that start near 0.17 at t = 0.01 and fall to about 0.05 to
0.07 at t = 0.50, lying close together; a treat-all line that starts with them and drops below 0 just
before t = 0.18; and the dashed treat-none line at 0.

The curves are close, so the next cell prints their values at eight thresholds. `[...] + [...]`
joins two lists end to end.

In [ ]:
# GIVEN — run as-is
# The decision curve's values at eight thresholds
SHOW_T = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.50]
print(f"{'t':>5}{'logistic':>11}{'forest':>11}{'boosted':>11}{'treat all':>11}")
for t in SHOW_T:
    row = [net_benefit(y_test, p, t) for p in probs.values()] + [net_benefit(y_test, treat_all, t)]
    print(f"{t:>5.2f}" + "".join(f"{value:>11.4f}" for value in row))

**Expected output:**
```
    t   logistic     forest    boosted  treat all
 0.02     0.1626     0.1620     0.1626     0.1609
 0.05     0.1483     0.1470     0.1466     0.1344
 0.08     0.1377     0.1372     0.1334     0.1062
 0.10     0.1303     0.1296     0.1249     0.0863
 0.15     0.1139     0.1198     0.1127     0.0325
 0.20     0.1027     0.1078     0.1046    -0.0279
 0.30     0.0906     0.0959     0.0921    -0.1748
 0.50     0.0540     0.0673     0.0700    -0.6447
```

### YOUR DIAGNOSIS

1. Explain why a 10 : 1 cost ratio gives a threshold of 1/11 and not 1/10, and how someone who names a threshold has stated a cost ratio without naming a dollar figure.
2. From the curve and the table: over which thresholds does each model have the highest net benefit, and is there a threshold between 0.01 and 0.50 at which some model fails to beat both references? What does that say about calling one model "best"?
3. At `T_RULE` the net benefit and the cost per row must rank the models in the same order: with these costs, cost per row = 10 × (prevalence − net benefit) at t = 1/11. Compare each model's cost per row here with its min cost in Diagnosis 1. What did the min cost column let each model do that `T_RULE` does not, and how can `logistic` come last in the three ranking columns and still be cheapest at 10 : 1?

*Your answers here:*

---

## Write it yourself (required — 30 min)

This function is required. No AI: write it yourself.

Diagnosis 2 put an interval on each model's score by itself. A ranking needs an interval on the
*gap* between two models, and because both models are scored on the same test rows, the gap has to
be resampled in pairs: each resample draws one set of rows and scores *both* models on those rows.
In the empty cell below, write a function

`paired_gap(y_true, prob_a, prob_b, metric_fn, boots=1000, seed=RANDOM_STATE)`

that returns a tuple of three numbers, `(gap, low, high)`:

1. `gap` is model a's score minus model b's score, from `metric_fn` on all the rows.
2. For each of `boots` resamples, draw n row positions the way `metric_interval` does, from a
   generator made once with `np.random.default_rng(seed)`, and record model a's score minus model
   b's score on those rows.
3. `low` and `high` are the 2.5th and 97.5th percentiles of the recorded gaps.

Start from `metric_interval` in Diagnosis 2: what changes is what each resample records and what the
function returns. Do not call `metric_interval` itself: it scores one model and returns its own
interval, not a gap's. `return gap, low, high` returns the three numbers as a tuple. You need
about ten lines.

The check runs your function on the forest and the boosted model with ROC AUC; then on the first
1,000 test rows with average precision, 400 resamples and a seed of 7; and it counts how many times
your function calls `metric_fn`. Until you write the function, the check says "Not written yet"; once
you do, it fails with a message until your function is right. The check takes a few seconds.

In [ ]:
# WRITE IT YOURSELF — required: write paired_gap(y_true, prob_a, prob_b, metric_fn, boots=1000, seed=RANDOM_STATE)

In [ ]:
# CHECK — for paired_gap above. Run it; you need not read it.
# globals() holds every name defined so far in this notebook
if "paired_gap" not in globals():
    print("Not written yet: write paired_gap in the empty cell above, then run this check again.")
else:
    import inspect
    from collections import Counter

    def names_used(code):
        """Every global or attribute name a function's code refers to, including inner functions."""
        names = set(code.co_names)
        for const in code.co_consts:
            if hasattr(const, "co_names"):
                names |= names_used(const)
        return names

    def near(a, b):
        """Equal to about six significant digits."""
        return bool(np.isclose(a, b, rtol=1e-6, atol=1e-9))

    # inspect.signature lists the names of a function's arguments, in order, with their defaults
    signature = inspect.signature(paired_gap)
    params = list(signature.parameters)
    ARGS = ["y_true", "prob_a", "prob_b", "metric_fn", "boots", "seed"]
    assert params[:6] == ARGS, (
        f"paired_gap's arguments are ({', '.join(params)}). The check calls it as the task asks: "
        "paired_gap(y_true, prob_a, prob_b, metric_fn, boots=1000, seed=RANDOM_STATE).")
    assert signature.parameters["boots"].default == 1000, "Give boots a default of 1000, as the task says."
    assert signature.parameters["seed"].default == RANDOM_STATE, (
        "Give seed a default of RANDOM_STATE, as the task says.")
    used = names_used(paired_gap.__code__)
    assert "metric_interval" not in used, (
        "paired_gap calls metric_interval, which scores one model and returns its own interval, not "
        "a gap's. Copy its loop instead, and score both models on each resample's rows.")
    assert not {"seed", "randint", "choice"} & used or "default_rng" in used, (
        "paired_gap uses np.random.seed or np.random's own functions, an older generator that draws "
        "other rows. Make one generator, gen = np.random.default_rng(seed), as metric_interval does.")
    assert "RANDOM_STATE" not in used, (
        "paired_gap uses RANDOM_STATE inside its body, so its seed argument changes nothing. Make the "
        "generator from the argument: np.random.default_rng(seed).")
    for lab_name in ["y", "y_test", "y_train", "X_test", "probs", "p", "models", "METRICS",
                     "auc_m", "value", "low", "high", "T_RULE"]:
        assert lab_name not in used, (
            f"paired_gap uses {lab_name}, a name from the lab, instead of its arguments. Compute "
            "everything from y_true, prob_a, prob_b and metric_fn.")

    # Diagnosis 1's test rows and probabilities, as Diagnosis 1 left them
    try:
        assert len(y_test) == 3000 and int(np.sum(y_test)) == 533
        for name in ["forest", "boosted", "logistic"]:
            assert np.allclose(probs[name], models[name].predict_proba(X_test)[:, 1])
    except Exception:
        raise AssertionError(
            "y_test or probs is no longer what Diagnosis 1 computed: a cell gave one of those names "
            "to something else. Give your own results other names, then rerun the notebook from the "
            "top.") from None

    def run(where, y_true, prob_a, prob_b, metric_fn, **options):
        """Call paired_gap on copies, counting metric_fn's calls; return (gap, low, high, calls)."""
        calls = []

        def counted(y_part, p_part):
            calls.append(np.asarray(y_part, dtype=float).tobytes())
            return metric_fn(y_part, p_part)

        try:
            got = paired_gap(y_true.copy(), prob_a.copy(), prob_b.copy(), counted, **options)
        except ValueError as err:
            text = str(err)
            if "continuous" in text:
                raise AssertionError(
                    f"{where}, the metric stopped with ValueError: {text}. That is what a metric says "
                    "when the probabilities arrive where the outcomes belong. Call it with the "
                    "outcomes first: metric_fn(y_true[rows], prob_a[rows]).") from None
            if "Quantiles must be" in text:
                raise AssertionError(
                    f"{where}, paired_gap stopped with ValueError: {text}. np.quantile takes shares "
                    "between 0 and 1; np.percentile takes percentages, as metric_interval uses "
                    "it.") from None
            raise AssertionError(f"{where}, paired_gap stopped with ValueError: {text}.") from None
        except IndexError as err:
            raise AssertionError(
                f"{where}, paired_gap stopped with IndexError: {err}. It drew a row position past "
                "the end of the data: does it use a fixed number of rows, such as 3000, instead of "
                "n = len(y_true)?") from None
        except Exception as err:
            raise AssertionError(f"{where}, paired_gap stopped with {type(err).__name__}: "
                                 f"{str(err).rstrip('.')}.") from None
        assert got is not None, f"{where}, paired_gap gives back nothing: use return, not print."
        if isinstance(got, dict):
            raise AssertionError(f"{where}, paired_gap gives back a dictionary with "
                                 f"{', '.join(repr(k) for k in got)}. Return a tuple: "
                                 "return gap, low, high.")
        if not isinstance(got, (tuple, list, np.ndarray)) or np.ndim(got) == 0:
            raise AssertionError(f"{where}, paired_gap gives back one {type(got).__name__}, not three "
                                 "numbers. Return a tuple: return gap, low, high.")
        n_boots = options.get("boots", 1000)
        assert len(got) != n_boots, (
            f"{where}, paired_gap gives back the {n_boots} resampled gaps. Return three numbers: the "
            "gap on all the rows, and the 2.5th and 97.5th percentiles of the resampled gaps.")
        if len(got) == 2 and np.ndim(got[1]) == 1:
            raise AssertionError(f"{where}, paired_gap gives back two things, the second an array: "
                                 "unpack the percentiles, low, high = np.percentile(...), and "
                                 "return gap, low, high.")
        assert len(got) == 3, (f"{where}, paired_gap gives back {len(got)} numbers. Return three: "
                               "return gap, low, high.")
        try:
            gap, low, high = (float(v) for v in got)
        except (TypeError, ValueError):
            raise AssertionError(f"{where}, paired_gap gives back three things that are not all "
                                 "single numbers. Return gap, low, high.") from None
        assert not np.isnan([gap, low, high]).any(), f"{where}, paired_gap gives back nan."
        return gap, low, high, calls

    # 1. the forest against the boosted model, ROC AUC, the default boots and seed
    where = "On the forest against the boosted model"
    full_y = y_test.astype(float).tobytes()
    gap, low, high, calls = run(where, y_test, probs["forest"], probs["boosted"], metrics.roc_auc_score)
    resampled = [c for c in calls if c != full_y]
    assert len(resampled) != 1000, (
        f"{where}, paired_gap scores only one model on each resample's rows ({len(resampled)} "
        "resampled scores for 1,000 resamples). Score both models on the same resampled rows.")
    assert len(resampled) != 1998, (
        f"{where}, paired_gap scores 999 resamples, not 1,000. Does its loop start at 1, "
        "range(1, boots)?")
    assert len(resampled) == 2000 or not resampled, (
        f"{where}, paired_gap scores a model on resampled rows {len(resampled)} times. With 1,000 "
        "resamples and two models that should be 2,000: one score per model per resample.")
    if resampled:
        pairs = Counter(resampled)
        assert sum(k == 1 for k in pairs.values()) < len(pairs) / 2, (
            f"{where}, paired_gap scores the two models on different rows: each resample draws one "
            "set of rows for model a and another for model b. Draw the rows once per resample and "
            "score both models on them.")
    assert not (near(low, high) and near(low, gap)), (
        f"{where}, paired_gap gives an interval of width 0: every resample holds the same rows in "
        "another order. Draw with replacement, as gen.integers(0, n, n) does.")
    assert not near(low, high), (
        f"{where}, paired_gap gives an interval of width 0: every resample draws the same rows. Make "
        "the generator once, before the loop, not inside it.")

    again = run("On a repeat call", y_test[:500], probs["forest"][:500], probs["boosted"][:500],
                metrics.roc_auc_score, boots=50, seed=3)
    twice = run("On a repeat call", y_test[:500], probs["forest"][:500], probs["boosted"][:500],
                metrics.roc_auc_score, boots=50, seed=3)
    assert again[:3] == twice[:3], (
        "Two calls with the same seed gave different intervals, so the function does not use its "
        "seed. Make the generator from it: gen = np.random.default_rng(seed).")

    MAIN = (0.0078347508, 0.0019882219, 0.0135130920)
    assert not (near(gap, MAIN[1]) and near(high, MAIN[0])), (
        f"{where}, paired_gap gives back the three numbers in the wrong order. Return the gap first, "
        "then the two ends of the interval: return gap, low, high.")
    assert not near(gap, -MAIN[0]), (
        f"{where}, paired_gap gives a gap of {gap:+.4f}; it should be {MAIN[0]:+.4f}. It subtracts "
        "the wrong way round: the gap is model a's score minus model b's.")
    assert not near(gap, 0.0076913356), (
        f"{where}, paired_gap gives a gap of {gap:+.4f}, the mean of the resampled gaps. The gap is "
        "model a's score minus model b's on all the rows.")
    assert near(gap, MAIN[0]), (
        f"{where}, paired_gap gives a gap of {gap:+.4f}; it should be {MAIN[0]:+.4f}, the forest's ROC "
        "AUC minus the boosted model's in Diagnosis 1's table (0.8981 - 0.8902, before rounding).")
    MAIN_SLIPS = {
        -0.0131778526: "come from different rows for the two models. Draw the rows once per resample "
                       "and score both models on them",
        -0.0120690826: "come from 1,000 resamples for model a and 1,000 others for model b. Draw the "
                       "rows once per resample and score both models on them",
        -0.0065354153: "score model b on all the rows in every resample. Score both models on the "
                       "resampled rows",
        -0.0106452642: "resample the probabilities but not the outcomes. Use y_true[rows] beside "
                       "prob_a[rows] and prob_b[rows]",
        -0.0135130920: "subtract the wrong way round inside the loop. Record model a's score minus "
                       "model b's",
        0.0028731137: "are the 5th and 95th percentiles. A 95% interval runs from the 2.5th to the "
                      "97.5th",
        -0.0011072803: "are the 0.025th and 0.975th percentiles: np.percentile takes percentages, so "
                       "ask for 2.5 and 97.5",
        0.0055769555: "are the 25th and 75th percentiles. A 95% interval runs from the 2.5th to the "
                      "97.5th",
        0.0019882318: "include the gap on all the rows among the resampled gaps. Record only the "
                      "boots resampled gaps, and keep the full-data gap for the first number",
        0.0022565024: "come from np.random.seed and np.random.randint. Make one generator, "
                      "gen = np.random.default_rng(seed), and draw with gen.integers",
    }
    for slip, cause in MAIN_SLIPS.items():
        assert not near(low, slip), (
            f"{where}, paired_gap gives the interval [{low:+.4f}, {high:+.4f}]. Your numbers {cause}.")
    assert near(low, MAIN[1]) and near(high, MAIN[2]), (
        f"{where}, paired_gap gives the interval [{low:+.6f}, {high:+.6f}]; it should be "
        f"[{MAIN[1]:+.6f}, {MAIN[2]:+.6f}]. Compare each step with the task: one generator made from "
        "seed, n row positions per resample drawn as metric_interval draws them, both models scored "
        "on those rows, and the 2.5th and 97.5th percentiles of the gaps. Another random draw (a new "
        "generator for each resample, or df.sample) gives a valid but different interval: use one gen "
        "and gen.integers.")

    # 2. the first 1,000 rows, logistic against forest, average precision, 400 resamples, seed 7
    where = "On the first 1,000 rows with boots=400 and seed=7"
    gap2, low2, high2, calls2 = run(where, y_test[:1000], probs["logistic"][:1000],
                                    probs["forest"][:1000], metrics.average_precision_score,
                                    boots=400, seed=7)
    resampled2 = [c for c in calls2 if c != y_test[:1000].astype(float).tobytes()]
    assert len(resampled2) == 800, (
        f"{where}, paired_gap scores a model on resampled rows {len(resampled2)} times; with 400 "
        "resamples it should be 800. Does its loop run range(boots), or a fixed count such as "
        "range(1000)?")
    VARIED = (-0.0498330265, -0.1042801230, 0.0058920079)
    assert near(gap2, VARIED[0]), (
        f"{where}, paired_gap gives a gap of {gap2:+.4f}; it should be {VARIED[0]:+.4f}. Does it use "
        "its metric_fn argument for the gap, on all the rows it was given?")
    assert not near(low2, -0.1007397007), (
        f"{where}, paired_gap gives [{low2:+.4f}, {high2:+.4f}], the interval for seed 42: it ignores "
        "its seed argument. Make the generator from it: np.random.default_rng(seed).")
    assert not near(low2, 0.0062441784), (
        f"{where}, paired_gap gives [{low2:+.4f}, {high2:+.4f}]: it records the size of each gap and "
        "drops the sign. Record model a's score minus model b's as it is.")
    assert near(low2, VARIED[1]) and near(high2, VARIED[2]), (
        f"{where}, paired_gap gives [{low2:+.6f}, {high2:+.6f}]; it should be [{VARIED[1]:+.6f}, "
        f"{VARIED[2]:+.6f}]. It passed the first test, so look for a number that does not come from "
        "the arguments: the rows, the resamples, the seed. Another random draw (a new generator for "
        "each resample, or df.sample) gives a valid but different interval: use one gen and "
        "gen.integers.")

    print(f"Passed: paired_gap matches. Forest minus boosted, ROC AUC: {gap:+.4f} "
          f"[{low:+.4f}, {high:+.4f}].")
    print(f"First 1,000 rows, logistic minus forest, AP (400 resamples, seed 7): {gap2:+.4f} "
          f"[{low2:+.4f}, {high2:+.4f}].")

---

## Ship It: `src/evaluation.py` (15 min)

Here is the module, finished except for one function. In it, every comparison of two models comes
with a paired interval on their gap. `metric_interval` and `net_benefit` are Diagnosis 2's and Diagnosis 3's functions. `gap_report`
takes two models' probabilities and a dictionary of metrics, and gives one row per metric: each
model's score, the gap, and your paired interval for the gap.

**Your task.** Paste your `paired_gap` from Write it yourself, unchanged, at the end of the first
cell, where it says so. Then run the two cells, compare the output with the expected output, and
answer the question under it. The first cell saves the module as a file; the second imports it and
compares two pairs of models on Diagnosis 1's four metrics and on the net benefit at `T_RULE`. It
takes about half a minute.

**Reading the cells.** `%%writefile`, the type notes (`boots: int`, `-> pd.DataFrame`), the import
and `hasattr` work as in Lab 17. The module sets its own `RANDOM_STATE = 42`, so your function's
default works there too. `pd.DataFrame(rows)` makes one table row per dictionary in the list `rows`,
and `.set_index("metric")` labels the rows. `{**METRICS, "net benefit": nb_at_rule}` is `METRICS` with
one entry more; `nb_at_rule` exists because a metric here takes only the outcomes and the
probabilities.

In [ ]:
%%writefile src/evaluation.py
"""Model evaluation in which every comparison of two models comes with a paired interval (ECON 5200 Lab 18).

A score from one test set invites a ranking the data may not support. Two models scored on the same
rows are compared by an interval on their gap, resampled in pairs, not by two separate intervals.
"""
import numpy as np
import pandas as pd

RANDOM_STATE = 42          # the default seed for every bootstrap


def metric_interval(y_true, y_prob, metric_fn, boots: int = 1000, seed: int = RANDOM_STATE) -> tuple:
    """The metric on all rows, and a 95% percentile bootstrap interval for it."""
    gen = np.random.default_rng(seed)
    n = len(y_true)
    values = []
    for _ in range(boots):
        rows = gen.integers(0, n, n)
        values.append(metric_fn(y_true[rows], y_prob[rows]))
    low, high = np.percentile(values, [2.5, 97.5])
    return metric_fn(y_true, y_prob), low, high


def net_benefit(y_true, y_prob, t: float) -> float:
    """True positives per row, minus false positives per row at the exchange rate t / (1 - t)."""
    flag = y_prob >= t
    n = len(y_true)
    return np.sum(flag & (y_true == 1)) / n - np.sum(flag & (y_true == 0)) / n * t / (1 - t)


def gap_report(y_true, prob_a, prob_b, metric_fns: dict, boots: int = 1000,
               seed: int = RANDOM_STATE) -> pd.DataFrame:
    """One row per metric: each model's score, the gap a - b, and its paired 95% interval."""
    rows = []
    for metric, fn in metric_fns.items():
        gap, low, high = paired_gap(y_true, prob_a, prob_b, fn, boots, seed)
        rows.append({"metric": metric, "a": fn(y_true, prob_a), "b": fn(y_true, prob_b),
                     "gap": gap, "low": low, "high": high})
    return pd.DataFrame(rows).set_index("metric")


# Paste your paired_gap function from Write it yourself below this line, unchanged.

In [ ]:
# GIVEN — run as-is, once your paired_gap is pasted into the module
# Import the module and compare two pairs of models, every gap with its paired interval
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import evaluation
importlib.reload(evaluation)       # reload, so a re-run of the %%writefile cell takes effect


def nb_at_rule(y_true, y_prob):
    """Diagnosis 3's net benefit at T_RULE, as a metric of the outcomes and probabilities alone."""
    return evaluation.net_benefit(y_true, y_prob, T_RULE)


if not hasattr(evaluation, "paired_gap"):
    print("Paste your paired_gap from Write it yourself into the module cell above first, "
          "then run that cell and this one again.")
else:
    REPORT = {**METRICS, "net benefit": nb_at_rule}
    for a, b in [("forest", "boosted"), ("logistic", "forest")]:
        print(f"a = {a}, b = {b}")
        print(evaluation.gap_report(y_test, probs[a], probs[b], REPORT).round(4).to_string())
        print()

**Expected output:**
```
a = forest, b = boosted
                  a       b     gap     low    high
metric                                             
accuracy     0.8897  0.8923 -0.0027 -0.0087  0.0033
ROC AUC      0.8981  0.8902  0.0078  0.0020  0.0135
AP           0.7152  0.7189 -0.0038 -0.0201  0.0125
min cost     0.4370  0.4667 -0.0297 -0.0597  0.0000
net benefit  0.1327  0.1277  0.0050  0.0004  0.0098

a = logistic, b = forest
                  a       b     gap     low    high
metric                                             
accuracy     0.8763  0.8897 -0.0133 -0.0230 -0.0037
ROC AUC      0.8898  0.8981 -0.0083 -0.0154 -0.0003
AP           0.6966  0.7152 -0.0185 -0.0456  0.0106
min cost     0.4347  0.4370 -0.0023 -0.0337  0.0347
net benefit  0.1339  0.1327  0.0012 -0.0019  0.0046
```
`net benefit` is at `T_RULE` = 1/11. For `min cost` and `net benefit`, remember which way is better:
a negative gap in cost favours model a, a negative gap in net benefit favours model b.

### YOUR DIAGNOSIS

1. Go back to Diagnosis 1's four winners. For each, say whether its paired interval supports it. Then compare the paired interval on the forest's ROC AUC lead over the boosted model with the two separate intervals in Diagnosis 2, explain why one is so much narrower, and say in two sentences what you would tell the team.

*Your answers here:*

---

## Extension (Optional — 20 min): Evaluating on the Wrong Population

The test rows came from the same population as the training rows. Suppose that in deployment the
first feature runs higher, with a mean of 1 instead of 0, while the way the features drive the
outcome stays the same: the same log-odds formula, region included.

**The task.** In the cell below, and in new cells under it:
1. Draw 3,000 new rows as Diagnosis 1 did, with a new generator, `np.random.default_rng(7)`, and add
   1 to the first column of the features before you compute the region and the log-odds.
2. Score the new rows with the three fitted models in `models`, and print the new prevalence and
   Diagnosis 1's four metrics for each model, beside Diagnosis 1's table.
3. Run your `paired_gap` on the forest and the boosted model with ROC AUC on the new rows, and compare
   it with the ROC AUC row of Ship It's first table.

In [ ]:
# YOUR TASK — optional: evaluation on a shifted population

### Interpretation Questions

1. Which metrics move most, and why? Separate what changed in the population (the prevalence, the share of rows in the region) from what changed in the models, which did not change at all.
2. Say what you would monitor in production to notice that your evaluation set has stopped representing the population you serve, before the outcomes arrive.

*Your answers here:*

1.

2.

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Scored three models (logistic regression, random forest, gradient
  boosting) four ways on one test set of 3,000 rows. Three different
  models came out best: boosting on accuracy ([YOUR VALUE]) and
  average precision ([YOUR VALUE]), the forest on ROC AUC
  ([YOUR VALUE]), and logistic regression on expected cost
  ([YOUR VALUE]) with a missed case costing 10 times a false alarm
* Put a bootstrap interval on every score: the forest's ROC AUC
  interval was [YOUR VALUE] to [YOUR VALUE]
* Used decision curve analysis, where the threshold 1/11 stands for
  the 10 : 1 cost ratio: net benefit [YOUR VALUE] for logistic
  regression and [YOUR VALUE] for treating everyone
* Wrote my own paired bootstrap for the gap between two models: the
  forest's ROC AUC lead over boosting was [YOUR VALUE], with an
  interval of [YOUR VALUE] to [YOUR VALUE]
* Shipped a module in which every comparison of two models comes
  with a paired interval

**Please write a README.md entry including:**
1. Project Title: Model Evaluation: A Metric That Chose Itself
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person, with no buzzwords. Use only
the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab18-model-evaluation`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab18-model-evaluation`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab18-model-evaluation`, branch `main`,
   commit message `Lab 18 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/evaluation.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.